# Kaggle Submission Generator
**F20AA CW2 — Google Maps Review Challenge**

Generates Kaggle-ready CSV using:
- TF-IDF **refitted on all 271,897 training rows** (bigrams, max_features=10000, min_df=2)
- Logistic Regression with best hyperparameters (C=1.0, L1) trained on full data
- Same preprocessing pipeline from Task 2

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Imports done.')

Imports done.


## 2. Preprocessing Pipeline (same as Task 2)

In [2]:
lemmatizer = WordNetLemmatizer()

STOPWORDS = set(stopwords.words('english'))
KEEP_NEGATIONS = {'not', 'never', 'no', 'nor', 'none'}
STOPWORDS -= KEEP_NEGATIONS

def remove_emojis(text):
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F9FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+', flags=re.UNICODE)
    return emoji_pattern.sub('', text)

def handle_negations(text):
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'cannot', text)
    return text

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = remove_emojis(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.lower()
    text = handle_negations(text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

sample = "I can't believe how TERRIBLE this place is! Never going back."
print('Input: ', sample)
print('Output:', preprocess(sample))

Input:  I can't believe how TERRIBLE this place is! Never going back.
Output: ca not believe terrible place never going back


## 3. Load & Preprocess ALL Training Data

In [3]:
print('Loading full training data...')
train = pd.read_csv('data/train_english.csv')
print(f'Train shape: {train.shape}')
train.head(2)

Loading full training data...
Train shape: (271897, 3)


,text,rating,text_length
0,This place is TERRIBLE; the people in charge a...,2,551
1,Terrible Service! And they are saying that I n...,1,258


In [4]:
print('Preprocessing all training texts...')
train['cleaned'] = train['text'].apply(preprocess)
print('Done.')

Preprocessing all training texts...
Done.


## 4. Refit TF-IDF on ALL Training Data

**Why:** The saved `tfidf_vectorizer.pkl` was fitted on only 80% of the data (217k rows). 
Refitting on all 271k rows gives the vectorizer a richer, more complete vocabulary — improving coverage on the Kaggle test set.

In [5]:
print('Fitting TF-IDF on ALL 271,897 training rows...')
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2
)
X_train = tfidf.fit_transform(train['cleaned'])
y_train = train['rating']

print(f'Feature matrix shape: {X_train.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')

Fitting TF-IDF on ALL 271,897 training rows...
Feature matrix shape: (271897, 50000)
Vocabulary size: 50,000


## 5. Train Logistic Regression on Full Data

In [6]:
print('Training Logistic Regression on full dataset...')
lr = LogisticRegression(
    C=1.0,
    penalty='l1',
    solver='liblinear',
    class_weight=None,
    max_iter=1000,
    random_state=42
)
lr.fit(X_train, y_train)
print('Model trained on full dataset.')

Training Logistic Regression on full dataset...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Model trained on full dataset.


## 6. Load Kaggle Test Data & Generate Predictions

In [7]:
print('Loading Kaggle test data...')
test = pd.read_csv('data/kaggle_test.csv')
test.columns = [c.strip().lower() for c in test.columns]
print(f'Test shape: {test.shape}')
print(f'Columns: {test.columns.tolist()}')
test.head()

Loading Kaggle test data...
Test shape: (89100, 2)
Columns: ['id', 'text']


,id,text
0,1,Quite easy to rent a car bur it is not easy to...
1,2,Nice voleyball court close to restaurants and ...
2,3,"Very nice built homes, the future locations ar..."
3,4,This dental clinic appears to be friendly with...
4,5,We came in to discuss tattoos. Only person th...


In [8]:
print('Preprocessing test texts...')
test['cleaned'] = test['text'].apply(preprocess)
print('Done.')

Preprocessing test texts...
Done.


In [9]:
print('Transforming and predicting...')
X_test = tfidf.transform(test['cleaned'])
predictions = lr.predict(X_test)

print(f'Predictions shape: {predictions.shape}')
print('\nPrediction distribution:')
print(pd.Series(predictions).value_counts().sort_index())

Transforming and predicting...
Predictions shape: (89100,)

Prediction distribution:
1    21934
2     3135
3     6388
4    51953
5     5690
Name: count, dtype: int64


## 7. Save Submission CSV

In [10]:
id_col = 'id' if 'id' in test.columns else test.columns[0]

submission = pd.DataFrame({
    'Id': test[id_col].values,
    'Rating': predictions
})

# Validate
assert submission.shape[1] == 2
assert list(submission.columns) == ['Id', 'Rating']
assert submission['Rating'].between(1, 5).all()
assert submission['Id'].is_unique
assert not submission.isnull().any().any()

print(f'All checks passed!')
print(f'Rows: {len(submission):,}')
submission.head(10)

All checks passed!
Rows: 89,100


,Id,Rating
0,1,4
1,2,4
2,3,4
3,4,4
4,5,5
5,6,4
6,7,4
7,8,4
8,9,1
9,10,4


In [11]:
output_path = 'data/kaggle_submission/submission_LR_fulldata.csv'
submission.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')

verify = pd.read_csv(output_path)
print(f'Verified rows: {len(verify):,}')
print(verify.head())

Saved to: data/kaggle_submission/submission_LR_fulldata.csv
Verified rows: 89,100
   Id  Rating
0   1       4
1   2       4
2   3       4
3   4       4
4   5       5
